> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 6 – Pré-processamento II: SNV e outliers por PCA

**Objetivo:** normalizar os espectros e remover amostras anômalas.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import carregar_dominio, preencher_nan, subamostra, cortar, suavizar, preparar_pastas, OUT_DIR, NIR_FILE
preparar_pastas()

In [ ]:
meta, X, wl = carregar_dominio(NIR_FILE)
meta, X = subamostra(meta, preencher_nan(X), n=500)
X, wl = cortar(X, wl, 450)
X = suavizar(X, wl)

## Exercício 1 – SNV
Para cada linha: subtraia a média da linha e divida pelo desvio padrão da linha. Use `keepdims=True`.

In [ ]:
def snv(X):
    return (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)

X_snv = snv(X)
print(np.allclose(X_snv.mean(1), 0), np.allclose(X_snv.std(1), 1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(wl, X[:100].T, alpha=0.2); ax[0].set_title('Antes do SNV')
ax[1].plot(wl, X_snv[:100].T, alpha=0.2); ax[1].set_title('Depois do SNV')
plt.show()

## Exercício 2 – Outliers por PCA (T² e Q)
Passos: centralizar → SVD → scores dos `k` primeiros PCs → T² (soma de score²/variância) → resíduo da reconstrução → Q (soma dos resíduos²). Outlier = T² ou Q acima do quantil `q`.

In [ ]:
def outliers_pca(X, k=8, q=0.99):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    T = U[:, :k] * S[:k]
    lam = (S[:k] ** 2) / (len(X) - 1)
    T2 = np.sum(T ** 2 / lam, axis=1)
    resid = Xc - T @ Vt[:k]
    Q = np.sum(resid ** 2, axis=1)
    return (T2 > np.quantile(T2, q)) | (Q > np.quantile(Q, q)), T2, Q

## Exercício 3 – Gráfico T² × Q e exportação

In [ ]:
out, T2, Q = outliers_pca(X_snv)
plt.scatter(T2[~out], Q[~out], s=10, label='normal')
plt.scatter(T2[out], Q[out], s=20, color='red', label='outlier')
plt.axvline(np.quantile(T2, 0.99), ls='--', color='gray')
plt.axhline(np.quantile(Q, 0.99), ls='--', color='gray')
plt.xlabel('T²'); plt.ylabel('Q'); plt.legend(); plt.show()
print(f'{out.sum()} outliers de {len(out)}')

limpo = pd.concat([meta.loc[~out].reset_index(drop=True),
                   pd.DataFrame(X_snv[~out], columns=[str(w) for w in wl])], axis=1)
limpo.to_csv(OUT_DIR / '02_nir_clean.csv', index=False)
pd.DataFrame({'T2': T2, 'Q': Q, 'outlier': out}).to_csv(OUT_DIR / '02_nir_outliers.csv', index=False)

## Checkpoint

Que problema real de laboratório pode gerar um Q alto com T² normal? E um T² alto com Q normal?